## 1. Импорты

In [ ]:
import os
import gc
import sys
import json
import toml
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import matplotlib.pyplot as plt
import tqdm
from prefect.blocks.system import Secret
from prefect.variables import Variable
import plotly.io as pio
from scipy.stats import linregress

import lightgbm as lgb
from sklearn.metrics import roc_auc_score

pio.renderers.default = "plotly_mimetype"

In [ ]:
!pip install hyperopt

In [ ]:
import lightgbm as lgb
from sklearn.feature_selection import RFE
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_pinball_loss
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

## 2. Параметры

In [ ]:
secret_cdw = await Secret.load("pass-sb-analytics")
secret_space = await Secret.load("pass-space")
secret_sb_dirkaim = (await Secret.load("pass-sb-dirkaim")).get()

In [ ]:
SUBTASK = 'v2_quantile'             # Подзадача (или версия, свободное наименование)
COMMENT = ""               # Текст-подсказка в свободной форме, пояснение для разработчика
SEED = 41                  # random_state

USERNAME_SPACE = "analytics"
USERNAME_CDW = "sb_analytics"
USERNAME_SB_DIRKAIM = secret_sb_dirkaim["login"]
PASSWORD_SPACE = secret_space.get()
PASSWORD_CDW = secret_cdw.get()
PASSWORD_SB_DIRKAIM = secret_sb_dirkaim["password"]


common_params = toml.load('common_params.toml')

AUTHOR = common_params['AUTHOR']        # Разработчик
PREFIX = common_params['PREFIX']        # Префикс
TASK_NUM = common_params['TASK_NUM']    # Номер задачи eservices
OUT_TABLE = common_params['OUT_TABLE']  # out_table
ML_TYPE = 'reg'      # Тип ML-задачи: reg - регрессия, clf - классификация clf, mclf - мультирег,mclf - мультикласс
TARGET = common_params['TARGET']        # Имя целевой переменной
TIME_COL = common_params['TIME_COL']    # Имя колонки с датами

PRIMARY_KEY = ['client_did', 'report_date', 'contract_uid']       # Объект предсказания, ключ

# Технические поля - не фичи, не id, не дата, для удаления
TECH_COLS = [
    'ub_clients_registration_dt_diff_report_date',
    'ub_clients_client_email',
    'ub_clients_mailing_consent',
    "card_report_date_diff_card_work_dt_aggmin", # ЛИК в фиче! 18.12.2025
    'transaction_total_trans_cnt_1m',
    'transaction_ratio_cnt_trans_1m_to_1m',
    'transaction_supermarkets_trans_cnt_12m'
]


# Пути (для чтения файлов)
PATH_TO_UTILS = common_params['PATH_TO_UTILS'] # Путь к кастомной библиотеке toolbox

# Пути (для сохранения файлов)
DATA_FOLDER = common_params['DATA_FOLDER']            # Папка с выборками
MODELS_FOLDER = common_params['MODELS_FOLDER']        # Файлы модели
FEATURES_FOLDER = common_params['FEATURES_FOLDER']    # Папка с фичами
REPORTS_FOLDER = f'{MODELS_FOLDER}/{SUBTASK}/report'  # Папка с отчетами

# Пути к файлам
PATH_TO_SAMPLE = f'./{DATA_FOLDER}/sample.parquet'    # Исходная выборка для разработки
PATH_MODELS_DATA = f"./{MODELS_FOLDER}/{SUBTASK}"     # Папка с файлми модели
QUANTILES = {"p95": 0.95, "p99": 0.99}


In [ ]:
print(PATH_TO_SAMPLE, FEATURES_FOLDER, MODELS_FOLDER)
print(PATH_MODELS_DATA)
print(REPORTS_FOLDER)

In [ ]:
# Создаем необходимые директории
os.makedirs(MODELS_FOLDER, exist_ok=True)
os.makedirs(PATH_MODELS_DATA, exist_ok=True)
os.makedirs(REPORTS_FOLDER, exist_ok=True)

### Загрузка кастомного модуля

In [ ]:
sys.path.append(PATH_TO_UTILS)

In [ ]:
!pip install openpyxl # для from toolbox.modeling_utils.report

In [ ]:
from toolbox.modeling_utils import preprocessing 
from toolbox.modeling_utils import report
from toolbox.modeling_utils import feature_selection
from toolbox.modeling_utils import report_generator
from toolbox import oracle

## 3. Логгирование

In [ ]:
# Логгирование
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt = "%Y-%m-%d %H:%M",
    handlers = [
        # logging.FileHandler("logfile.log"),
        logging.StreamHandler(),
    ])

In [ ]:
!pip install clearml

In [ ]:
os.environ["CLEARML_AUTO_LOG_ARTIFACTS"] = "0" 
os.environ["CLEARML_MONITOR_SYSTEM"]    = "0"
os.environ["CLEARML_AUTO_CONNECT_FRAMEWORKS"] = "0"

import clearml
from clearml import Task, Logger, Model

PROJECT_NAME = f"eservices-{TASK_NUM}"

clearml_task = Task.init(
    project_name=PROJECT_NAME,
    task_name=f"{PROJECT_NAME}_{SUBTASK}",
    auto_connect_streams=False,
    reuse_last_task_id=False
)

## 4 Чтение данных

In [ ]:
filenames = list(filter(
    lambda file: file.startswith(PREFIX.lower()) and file.endswith('.parquet'),
    os.listdir(FEATURES_FOLDER)
))
feature_paths = [f'{FEATURES_FOLDER}/{filename}' for filename in filenames]

sample = pd.read_parquet(PATH_TO_SAMPLE)

In [ ]:
sample = sample[sample['report_date']>='2025-09-01']

In [ ]:
sample

## 4. EDA

In [ ]:
# Динамика таргета во времени
sample['year_month'] = sample[TIME_COL].dt.strftime('%Y-%m-01')
sample['year_month'] = pd.to_datetime(sample['year_month'])

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
display(sample.pivot_table(values=['client_did'],
                index=['year_month'],
                aggfunc='count',
                margins=False).T)

pt = (sample
    .pivot_table(
        values=["client_did"],
        index=["year_month"],
        aggfunc="count",
        margins=False,).reset_index())

fig_sample = px.bar(
    pt,
    x="year_month",
    y="client_did",
    orientation="v",
    title="Количество клиентов по месяцам",
)

fig_sample.show()  # в ноутбуке — опционально

In [ ]:
Logger.current_logger().report_plotly(
    title="001 Кол-во клиентов в выборке по месяцам",
    series="",
    figure=fig_sample)

In [ ]:
display(sample.pivot_table(values=['contract_uid'],
                index=['year_month'],
                aggfunc='count',
                margins=False).T)

pt = (sample
    .pivot_table(
        values=["contract_uid"],
        index=["year_month"],
        aggfunc="count",
        margins=False,).reset_index())

fig_sample = px.bar(
    pt,
    x="year_month",
    y="contract_uid",
    orientation="v",
    title="Количество контрактов по месяцам",
)

fig_sample.show()  # в ноутбуке — опционально

In [ ]:
Logger.current_logger().report_plotly(
    title="001 Кол-во контрактов в выборке по месяцам",
    series="",
    figure=fig_sample)

In [ ]:
from typing import Tuple, Optional, Sequence, List, Any
def counts_target_ratio_sample(
    data: pd.DataFrame,
    date_col: str,
    target_col: str,
    title: str = "Размеры и mean Target по месяцам",
    count_series_name: str = "Target count",
    rate_series_name: str = "Target mean",
    show: bool = True,
) -> Tuple[pd.DataFrame, go.Figure]:

    df = data.copy()
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col])

    # месяц = начало месяца
    df["month"] = df[date_col].dt.to_period("M").dt.to_timestamp()

    # count = число строк; rate = среднее target (доля)
    df_agg = (
        df.groupby("month", as_index=False)
        .agg(
            count=(target_col, "size"),
            # sum=(target_col, "sum"),
            above_zero=(target_col, lambda s: (s > 0).sum()),
            # rate=(target_col, "mean"),
            rate=(target_col, lambda s: round(pd.to_numeric(s, errors="coerce").mean(), 2)),
        )
        .sort_values("month")
    )

    bar_text = df_agg["count"]
    line_text = [
        f"{int(round(r))}" if pd.notna(r) else "" for r in df_agg["rate"]
    ]

    ymax = float(df_agg["count"].max() or 0)
    pad = ymax * 0.1  # запас сверху
    y2max = float(df_agg["rate"].max() or 0)
    pad2 = y2max * 0.25  # запас сверху

    fig = go.Figure()

    # БАРЫ с подписями сверху
    fig.add_bar(
        x=df_agg["month"],
        y=df_agg["count"],
        name=count_series_name,
        text=bar_text,
        textposition="outside",  # подписи над столбцами
        textfont=dict(size=12),
        marker=dict(line=dict(width=0)),
    )

    # ЛИНИЯ таргета (правая ось, проценты)
    fig.add_trace(
        go.Scatter(
            x=df_agg["month"],
            y=df_agg["rate"],
            mode="lines+markers+text",
            name=rate_series_name,
            yaxis="y2",
            text=line_text,
            textposition="top center",
            line=dict(width=3, color="#D62728"),
            marker=dict(size=8, line=dict(width=1, color="white")),
        )
    )

    fig.update_layout(
        title=title,
        xaxis=dict(title=None, tickangle=45),
        # верхний запас по оси Y, чтобы видно было подписи «outside»
        yaxis=dict(
            title="Количество контрактов", tickformat=",.0f", range=[0, ymax + pad]
        ),
        yaxis2=dict(
            title="mean(ОД)",
            overlaying="y",
            side="right",
            # tickformat=".0%",
            range=[0, y2max + pad2],
        ),
        bargap=0.25,
        template="plotly_white",
        legend=dict(orientation="h", yanchor="top", y=-0.35, xanchor="right", x=0.5),
        margin=dict(t=70, r=60, b=70, l=70),
    )

    if show:
        fig.show()

    df_agg["mean"] = (df_agg["rate"] ).round(2).astype(
        str
    ) + ""  # переведем в проценты

    return df_agg, fig

In [ ]:
info_sample_try, fig_sample_try = counts_target_ratio_sample(sample, 'report_date', 'target')

In [ ]:
info_sample_try[['month', 'count', 'mean']]

### Таргет по месяцам (аналог уровня таргета в классификации)
Среднее и медиана целевой переменной по месяцам — видно тренд и сезонность.

In [ ]:
def target_by_month(df, date_col, target_cols, agg="mean", figsize=(12, 4), show=True):
    """График агрегата таргета по месяцам (agg: 'mean' или 'median')."""
    if isinstance(target_cols, str):
        target_cols = [target_cols]

    df = df.copy()
    df["_month"] = df[date_col].dt.to_period("M")
    g = df.groupby("_month", observed=True)[target_cols].agg(agg)

    plot_df = g.reset_index()
    plot_df["_month"] = plot_df["_month"].astype(str)

    fig = px.line(
        plot_df,
        x="_month",
        y=target_cols,
        title=f"Таргет по месяцам ({agg})",
        markers=True,
    )

    fig.update_layout(
        width=int(figsize[0] * 80),
        height=int(figsize[1] * 80),
        yaxis_title=agg,
        legend=dict(
            yanchor="top",
            y=1,
            xanchor="left",
            x=1.02,
        ),
        margin=dict(r=120),
    )
    fig.update_xaxes(tickangle=45)

    if show:
        fig.show()

    return g, fig

In [ ]:
g, fig = target_by_month(sample, "report_date", TARGET, agg="mean")

In [ ]:
Logger.current_logger().report_plotly(
    title="002 Таргет по месяцам",
    series="",
    figure=fig,
)

### Распределение таргета
Гистограмма и ящик с усами — асимметрия, выбросы, нули.

In [ ]:
from plotly.subplots import make_subplots

In [ ]:
def _target_grid(n, ncols):
    nrows = (n + ncols - 1) // ncols
    return nrows, ncols


def _default_figsize(ncols_plot, nrows, aspect=1.2):
    return (4 * ncols_plot, max(4, 3 * nrows * aspect))


def target_histogram(
    df,
    target_cols,
    ncols=2,
    log_scale=False,
    bins=30,
    figsize=None,
    show=True,
):
    """Гистограммы по списку таргетов (отдельная фигура для ClearML)."""
    if isinstance(target_cols, str):
        target_cols = [target_cols]

    n = len(target_cols)
    nrows, ncols_grid = _target_grid(n, ncols)

    if figsize is None:
        figsize = _default_figsize(ncols, nrows)

    fig = make_subplots(
        rows=nrows,
        cols=ncols_grid,
        subplot_titles=list(target_cols),
        horizontal_spacing=0.1,
        vertical_spacing=0.15 if nrows > 1 else 0.12,
    )

    for i, col in enumerate(target_cols):
        r = i // ncols + 1
        c = i % ncols + 1

        x = df[col].dropna().astype(float)
        x_hist = np.log1p(x) if log_scale and x.max() > 0 else x

        counts, edges = np.histogram(x_hist, bins=bins)
        centers = (edges[:-1] + edges[1:]) / 2
        widths = np.diff(edges)

        fig.add_trace(
            go.Bar(
                x=centers,
                y=counts,
                width=widths,
                marker=dict(line=dict(color="black", width=0.5)),
                opacity=0.7,
                showlegend=False,
                name=col,
                hovertemplate="значение: %{x:.4g}<br>число: %{y}<extra></extra>",
            ),
            row=r,
            col=c,
        )

    fig.update_layout(
        width=int(figsize[0] * 80),
        height=int(figsize[1] * 80),
        autosize=False,
        title_text="Гистограмма таргета",
        margin=dict(l=50, r=30, t=70, b=50),
        bargap=0,
    )

    if show:
        fig.show()
    return fig

In [ ]:
fig_hist = target_histogram(sample, [TARGET], log_scale=True, figsize=(10, 5), show=True)

In [ ]:
Logger.current_logger().report_plotly(
    title="003 Распределение таргета",
    series="",
    figure=fig_hist,
)

In [ ]:
def target_boxplot(
    df,
    target_cols,
    ncols=2,
    figsize=None,
    show=True,
):
    if isinstance(target_cols, str):
        target_cols = [target_cols]
    n = len(target_cols)
    nrows, ncols_grid = _target_grid(n, ncols)
    if figsize is None:
        figsize = (3 * ncols_grid, max(3.5, 3 * nrows))
    fig, axes = plt.subplots(nrows, ncols_grid, figsize=figsize, squeeze=False)
    for i, col in enumerate(target_cols):
        r, c = i // ncols, i % ncols
        x = df[col].dropna().astype(float)
        axes[r, c].boxplot(x, vert=False)
        axes[r, c].set_title(f"{col} (box)")
        axes[r, c].set_xlabel(col)
    for j in range(n, nrows * ncols_grid):
        axes.flat[j].set_visible(False)
    fig.suptitle("Boxplot таргета", y=1.02)
    plt.tight_layout()
    if show:
        plt.show()
    return fig

In [ ]:
fig_box = target_boxplot(sample, [TARGET], ncols=1, figsize=(8, 5), show=True)

In [ ]:
Logger.current_logger().report_matplotlib_figure(
    title="004 Boxplot таргет",
    series="",
    figure=fig_box,
    iteration=0,
)

### Сводка по таргетам: среднее, медиана, std, доля нулей
Таблица — общая и при необходимости по месяцам.

In [ ]:
def target_summary(df, target_cols, by_month=False, date_col=None):
    """Таблица: count, mean, median, std, min, max, доля нулей. by_month=True — то же по месяцам."""
    df = df.copy()
    df[target_cols] = df[target_cols].astype(float)
    if by_month and date_col:   
        df["_month"] = df[date_col].dt.to_period("M")
        return df.groupby("_month")[target_cols].agg(
            ["count", "mean", "median", "std", "min", "max"]
        ).round(2)
    res = df[target_cols].agg(["count", "mean", "median", "std", "min", "max"]).T
    res["pct_zero"] = (df[target_cols] == 0).mean() * 100
    return res.round(2)

In [ ]:
target_summary(sample, [TARGET])

In [ ]:
tbl = target_summary(sample, [TARGET], by_month=True, date_col='report_date')

In [ ]:
tbl = tbl.T.droplevel(0).rename_axis("metric").reset_index()
tbl.columns.name = None
tbl.columns = [str(c) for c in tbl.columns]
tbl

In [ ]:
Logger.current_logger().report_table(
    title="005 Размер выбороки/статистика по таргету",
    series=" ",
    table_plot=tbl
)

In [ ]:
tbl.T.to_excel('regression_rep/check.xlsx')

In [ ]:
sample['report_date'].unique()

## 5. Собираем ВДР и фичи вместе

In [ ]:
merged_df = sample.copy() 

In [ ]:
merged_df = merged_df.drop(['current_limit', 'contract_src_id'], axis=1)

In [ ]:
%%time
import gc

MISSING_THRESHOLD = 0.8

for path, filename in tqdm.tqdm(zip(feature_paths, filenames), total=len(feature_paths)):
    feat_df = pd.read_parquet(path)

    merge_key = [c for c in PRIMARY_KEY if c in feat_df.columns]
    feat_df = feat_df.drop_duplicates(subset=merge_key, keep='first')

    missing_df = preprocessing.analyze_missing_values(
        feat_df, missing_threshold=MISSING_THRESHOLD
    )
    if missing_df is not None and not missing_df.empty:
        missing_df.to_excel(f'{REPORTS_FOLDER}/{filename}_missing.xlsx')
        feat_df = preprocessing.remove_columns_by_missing(feat_df, missing_df, 0.8)

    merged_df = merged_df.merge(feat_df, on=merge_key, how='left', copy=False)

    del feat_df, missing_df
    gc.collect()

In [ ]:
merged_df.info()
merged_df.shape

In [ ]:
merged_df.drop(columns=TECH_COLS, inplace=True, errors="ignore")

In [ ]:
display(merged_df.columns[:10])
all_features = merged_df.columns[4:] # Список всех доступных фичей

In [ ]:
gc.collect()

### Проверка

In [ ]:
display(merged_df.shape)
display(merged_df.head(3))

## 6. Отбор фичей (однофакторный анализ)


1. Убираем фичи с большой долей пропущенных значений
2. Убираем константные фичи
3. Убираем нестабильные фичи (`PSI > threshold`)
4. Разбиение на сегменты: TRAIN, TEST, OOT
5. Грубый отбор фичей (Оставляем ТОП-N фичей)

### 6.1 Убираем фичи с большой долей пропусков

In [ ]:
MISSING_THRESHOLD = 0.8  # Доля пропусков не более чем

missing_df = preprocessing.analyze_missing_values(merged_df, missing_threshold=MISSING_THRESHOLD)
missing_df.to_excel(f'{REPORTS_FOLDER}/missing_percentage.xlsx')

In [ ]:
merged_df = preprocessing.remove_columns_by_missing(merged_df, missing_df, 0.8)

In [ ]:
merged_df.shape

### 6.2 Убираем фичи с константными значениями (низкая дисперсия данных)

In [ ]:
res = preprocessing.find_constant_features(merged_df, threshold=0.95, verbose=False)
res['constant']

In [ ]:
merged_df.drop(res['constant'], axis = 1, inplace =True)

In [ ]:
merged_df.shape

### 6.3 Отбор по PSI

In [ ]:
features = list(merged_df.drop(['report_date', 'client_did', 'contract_uid', 'target_50', 'target_75', 'target_25', TARGET, 'year_month'], axis=1).columns)

In [ ]:
features

In [ ]:
%%time
psi_result = preprocessing.calculate_feat_psi(merged_df[list(features) + ['year_month']], features, date_col='year_month')

In [ ]:
psi_result.head(20)

In [ ]:
# Фильтр 1: глобальное ограничение PSI на всей выборке
PSI_THRESHOLD = 0.15 # Убираем фичи, с PSI > PSI_THRESHOLD
# Фильтр 2: Ограничение тренда
# Если есть линейный тренд на рост PSI, то максимальный тренд PSI не более PSI_TREND
PSI_TREND = 0.1
# PSI_TREND = 0.1

In [ ]:
to_del_all = []
for col in psi_result.columns:
    max_psi = psi_result[col].max()
    lrgs = linregress(range(len(psi_result[col])), psi_result[col])
    if ((lrgs.slope > 0) & (lrgs.pvalue < 0.05) & (psi_result[col].max() > PSI_TREND)) or psi_result[col].max() >= PSI_THRESHOLD:
        to_del_all.append(col)

In [ ]:
psi_result[to_del_all].to_excel(f'{REPORTS_FOLDER}/psi_to_delete.xlsx')

In [ ]:
len(to_del_all), merged_df.shape

In [ ]:
psi_result[ [   
    'current_limit',
    'contract_init_amt',
    'current_debt',
    # 'current_amt',
    'card_age_months',
    'current_interest_rate',
    'prol_prc_rt',
    'prc_rt_diff',
    'contract_amt_diff_init',
    'available_limit_by_debt',
    'util_current',
    'debt_daily_max_90d',
    'debt_daily_max_180d',
    'util_daily_max_90d',
    'util_daily_max_180d',
    'util_monthavg_avg_3m',
    'util_monthavg_avg_6m',
    'util_monthavg_avg_12m',
    'days_since_last_transaction',
	'risk_grade_last']]

In [ ]:
merged_df.drop(to_del_all, axis=1, inplace=True)

In [ ]:
merged_df.shape

In [ ]:
merged_df['target'] = merged_df['target'].astype(float)

### 6.4 Разбиение на сегменты: Train test split

Сегменты:
* `train` - Обучающая часть
* `test` - тестовая часть
* `oot` - в зависимости от типа валидации
* `other` - дубли, некорректные записи и тд

In [ ]:
df = merged_df[merged_df['report_date'] < '2026.02.01']
oot = merged_df[merged_df['report_date'] >= '2026.02.01']
train = df[df['report_date'] < '2026.01.01']
test = df[df['report_date'] >= '2026.01.01']

In [ ]:
train.shape, test.shape, oot.shape

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from typing import Tuple, Optional, Sequence, List, Any

def counts_target_ratio(
    train: pd.DataFrame,
    test: pd.DataFrame,
    oot: pd.DataFrame = None,
    date_col: str = "month",
    target_col: str = "target",
    id_col: str = None,
    title: str = "Размеры выборок и уровень таргета по месяцам",
) -> Tuple[pd.DataFrame, go.Figure]:
    """
    Строит график с размерами выборок train/test/oot и уровнем таргета по месяцам.

    Параметры:
    ----------
    train : pd.DataFrame
        Обучающая выборка
    test : pd.DataFrame
        Тестовая выборка
    oot : pd.DataFrame, optional
        OOT выборка (по умолчанию None)
    date_col : str, optional
        Название колонки с датой (по умолчанию "month")
    target_col : str, optional
        Название колонки с таргетом (по умолчанию "target")
    id_col : str, optional
        Название колонки с ID для подсчета уникальных значений (по умолчанию None)
    title : str, optional
        Заголовок графика (по умолчанию "Размеры выборок и уровень таргета по месяцам")

    Возвращает:
    -----------
    Tuple[pd.DataFrame, go.Figure]
        Агрегированные данные и Plotly фигура
    """

    def _prep(df: pd.DataFrame):
        if df is None or len(df) == 0:
            return pd.DataFrame(columns=["month", "cnt", "rr"])
        tmp = df.copy()
        tmp[date_col] = pd.to_datetime(tmp[date_col])
        tmp["month"] = tmp[date_col].dt.to_period("M").dt.to_timestamp()
        if id_col:
            g = tmp.groupby("month", as_index=False).agg(
                cnt=(id_col, pd.Series.nunique), rr=(target_col, "mean")
            )
        else:
            g = tmp.groupby("month", as_index=False).agg(
                cnt=(target_col, "size"), rr=(target_col, "mean")
            )
        return g

    g_train = _prep(train).rename(columns={"cnt": "train_cnt", "rr": "rr_train"})
    g_test = _prep(test).rename(columns={"cnt": "test_cnt", "rr": "rr_test"})
    g_oot = _prep(oot).rename(columns={"cnt": "oot_cnt", "rr": "rr_oot"})

    # полная сетка месяцев
    months = pd.DataFrame(
        {
            "month": sorted(
                set(g_train["month"]) | set(g_test["month"]) | set(g_oot["month"])
            )
        }
    )
    df = (
        months.merge(g_train, on="month", how="left")
        .merge(g_test, on="month", how="left")
        .merge(g_oot, on="month", how="left")
    )

    df = df[
        ["month", "train_cnt", "test_cnt", "oot_cnt", "rr_train", "rr_test", "rr_oot"]
    ]

    # NaN → 0 только для счетчиков
    for c in ["train_cnt", "test_cnt", "oot_cnt"]:
        if c in df:
            df[c] = df[c].fillna(0).astype(int)

    # --- маски для линий (None, где счетчик = 0) ---
    rr_train_y = [
        v  if (pd.notna(v) and cnt > 0) else None
        for v, cnt in zip(df.get("rr_train"), df.get("train_cnt", 0))
    ]
    rr_test_y = [
        v  if (pd.notna(v) and cnt > 0) else None
        for v, cnt in zip(df.get("rr_test"), df.get("test_cnt", 0))
    ]
    rr_oot_y = [
        v  if (pd.notna(v) and cnt > 0) else None
        for v, cnt in zip(df.get("rr_oot"), df.get("oot_cnt", 0))
    ]

    fig = go.Figure()

    # stacked bars: train + test (+ oot где есть)
    fig.add_bar(x=df["month"], y=df["train_cnt"], name="train_cnt")
    fig.add_bar(x=df["month"], y=df["test_cnt"], name="test_cnt")
    fig.add_bar(x=df["month"], y=df["oot_cnt"], name="OOT_cnt")

    # lines on secondary axis, masked by nonzero counts
    fig.add_trace(
        go.Scatter(
            x=df["month"],
            y=rr_train_y,
            mode="lines+markers",
            name="RR Train",
            yaxis="y2",
            line=dict(color="#17BECF"),
        )
    )
    fig.add_trace(
        go.Scatter(
            x=df["month"],
            y=rr_test_y,
            mode="lines+markers",
            name="RR Test",
            yaxis="y2",
            line=dict(color="#FFD200"),
        )
    )
    fig.add_trace(
        go.Scatter(
            x=df["month"],
            y=rr_oot_y,
            mode="lines+markers",
            name="RR OOT",
            yaxis="y2",
            line=dict(color="#7B61FF"),
        )
    )

    fig.update_layout(
        barmode="stack",
        title=title,
        xaxis=dict(title="Месяц"),
        yaxis=dict(title="Количество клиентов", tickformat=",.0f"),
        yaxis2=dict(title="RR, %", overlaying="y", side="right"),
        legend=dict(orientation="h", yanchor="top", y=-0.35, xanchor="center", x=0.5),
        template="plotly_white",
    )

    # показать и вернуть
    fig.show()

    df = df.fillna(0)
    df["rr_train"] = (df["rr_train"] ).round(2).astype(
        str
    )   # переведем в проценты
    df["rr_test"] = (df["rr_test"] ).round(2).astype(
        str
    )   # переведем в проценты
    df["rr_oot"] = (df["rr_oot"] ).round(2).astype(
        str
    )   # переведем в проценты
    df = df.replace("0.0", "---")
    df = df.replace(0, "---")
    return df, fig

In [ ]:
info_df, fig_df = counts_target_ratio(train, test, oot, TIME_COL, TARGET)
fig_df.update_layout(
    title="Размеры выборок и средний ОД по месяцам",
    yaxis2=dict(title="ОД", overlaying="y", side="right"),
)
for tr in fig_df.data:
    if tr.name and tr.name.startswith("RR "):
        tr.name = tr.name.replace("RR ", "ОД ")

info_df = info_df.rename(columns={
    "rr_train": "util_train",
    "rr_test": "util_test",
    "rr_oot": "util_oot",
})
fig_df

In [ ]:
Logger.current_logger().report_plotly(
    title="006 Размеры выборок и средний ОД по месяцам",
    series="",
    figure=fig_df)

In [ ]:
info_df = info_df.rename(columns={
    "util_train": "mean(target) train",
    "util_test": "mean(target) test",
    "util_oot": "mean(target) oot",
})
info_df

In [ ]:
info_df_to_log = info_df.copy()
info_df_to_log["month"] = pd.to_datetime(info_df_to_log["month"]).dt.strftime("%Y-%m-%d")
Logger.current_logger().report_table(
    title="007 Размеры выборок и средний таргет по месяцам",
    series="",
    table_plot=info_df_to_log
)
info_df

In [ ]:
# Проверка на потерянные строки 
assert train[PRIMARY_KEY].shape[0] +\
        test[PRIMARY_KEY].shape[0] +\
         oot[PRIMARY_KEY].shape[0] == sample.dropna(subset=[TARGET]).shape[0]

## 7. Предварительный отбор фичей

In [ ]:
# Доработать
categorical_features = list(merged_df
                            .drop(columns=PRIMARY_KEY+['year_month','target_25','target_50','target_75',TARGET])
                            .select_dtypes(['category', 'object'])
                            .columns)
numerical_features = list(merged_df
                          .drop(columns=categorical_features + PRIMARY_KEY + ['year_month','target_25','target_50','target_75',TARGET])
                          .columns)

print(f'Categorical: {len(categorical_features)}')
print(f'Numerical: {len(numerical_features)}')

In [ ]:
features = categorical_features + numerical_features

In [ ]:
features

## 7. Квантильные модели P95 и P99
Один исходный таргет, две независимые модели. Все этапы отбора и настройки используют соответствующий pinball loss. `test` — валидация для настройки, OOT — итоговая проверка. Временные границы сохранены из исходного шаблона.

Прогнозы сохраняются без перестановки квантилей: пересечения явно показаны в отчёте. Gap — ширина верхнего хвоста, а не доверительный интервал. Прогноз средней задолженности не является автоматически готовым кредитным лимитом.

In [ ]:
def make_quantile_model(alpha, **params):
    defaults = dict(n_estimators=100, max_depth=4, random_state=SEED,
                    verbosity=-1, importance_type='gain')
    defaults.update(params)
    return lgb.LGBMRegressor(objective='quantile', alpha=alpha,
                             metric='quantile', **defaults)


def select_by_importance(data, target_col, eval_data, alpha, max_features=150):
    cols = data.drop(columns=[target_col]).select_dtypes(include=[np.number]).columns.tolist()
    model = make_quantile_model(alpha)
    model.fit(data[cols], data[target_col], eval_set=[(eval_data[cols], eval_data[target_col])])
    imp = pd.Series(model.feature_importances_, index=cols).sort_values(ascending=False)
    selected = imp.head(max_features).index.tolist()
    return selected, imp, model


def metric_vs_n_features(data, target_col, eval_data, imp, alpha, step=10, threshold=0.01):
    n_all = len(imp)
    if not n_all:
        raise ValueError('Нет признаков для обучения')
    sizes = sorted(set(list(range(min(20, n_all), n_all + 1, step)) + [n_all]))
    rows = []
    for size in sizes:
        cols = imp.head(size).index.tolist()
        model = make_quantile_model(alpha)
        model.fit(data[cols], data[target_col])
        loss = mean_pinball_loss(eval_data[target_col], model.predict(eval_data[cols]), alpha=alpha)
        rows.append(dict(n_features=size, pinball_loss=loss))
    table = pd.DataFrame(rows)
    cutoff = table.pinball_loss.min() * (1 + threshold)
    best_n = int(table.loc[table.pinball_loss <= cutoff, 'n_features'].min())
    return table, best_n, imp.head(best_n).index.tolist()


def tune_hyperparams(data, target_col, eval_data, alpha, max_evals=30):
    cols = [c for c in data if c != target_col]
    space = {
        'n_estimators': hp.quniform('n_estimators', 50, 300, 25),
        'max_depth': hp.quniform('max_depth', 3, 10, 1),
        'learning_rate': hp.loguniform('learning_rate', np.log(0.02), np.log(0.3)),
        'num_leaves': hp.quniform('num_leaves', 15, 127, 10),
        'min_child_samples': hp.quniform('min_child_samples', 5, 50, 5),
    }
    def cast(params):
        return {k: float(v) if k == 'learning_rate' else int(v) for k, v in params.items()}
    def objective(params):
        model = make_quantile_model(alpha, **cast(params))
        model.fit(data[cols], data[target_col])
        loss = mean_pinball_loss(eval_data[target_col], model.predict(eval_data[cols]), alpha=alpha)
        return {'loss': loss, 'status': STATUS_OK}
    trials = Trials()
    best = fmin(objective, space, algo=tpe.suggest, max_evals=max_evals,
                trials=trials, rstate=np.random.default_rng(SEED))
    params = cast(best)
    model = make_quantile_model(alpha, **params)
    model.fit(data[cols], data[target_col])
    return model, params, pd.DataFrame([
        {'trial': t['tid'], 'pinball_loss': t['result']['loss']} for t in trials.trials])

In [ ]:
for name, frame in [('train', train), ('test', test), ('oot', oot)]:
    if frame.empty:
        raise ValueError(f'{name}: пустая выборка')
    if not np.isfinite(frame[TARGET].to_numpy(dtype=float)).all():
        raise ValueError(f'{name}: таргет содержит NaN/inf')
    if frame.duplicated(PRIMARY_KEY).any():
        raise ValueError(f'{name}: дубли PRIMARY_KEY')

PRESELECTION_TOP_N = 150
results = {}
for label, alpha in QUANTILES.items():
    selected, imp, _ = select_by_importance(train[features + [TARGET]], TARGET,
        test[features + [TARGET]], alpha, PRESELECTION_TOP_N)
    report_n, best_n, selected = metric_vs_n_features(train, TARGET, test, imp[selected], alpha)
    model, params, trials = tune_hyperparams(train[selected + [TARGET]], TARGET,
        test[selected + [TARGET]], alpha, max_evals=30)
    results[label] = dict(alpha=alpha, model=model, selected=selected,
        best_n=best_n, best_params=params, metric_vs_n=report_n, trials=trials,
        baseline=float(train[TARGET].quantile(alpha)))
    display(report_n)

## 8. Прогнозы и квантильные метрики

In [ ]:
def quantile_metrics(y, pred, alpha, baseline):
    y, pred = np.asarray(y, dtype=float), np.asarray(pred, dtype=float)
    loss = mean_pinball_loss(y, pred, alpha=alpha)
    baseline_loss = mean_pinball_loss(y, np.full(len(y), baseline), alpha=alpha)
    excess = np.maximum(y - pred, 0)
    exceeded = y > pred
    coverage = np.mean(y <= pred)
    return dict(N=len(y), alpha=alpha, pinball_loss=loss,
        baseline_pinball_loss=baseline_loss,
        pinball_skill=1-loss/baseline_loss if baseline_loss > 0 else np.nan,
        coverage=coverage, coverage_gap_pp=100*(coverage-alpha),
        exceedance_rate=exceeded.mean(), exceedance_count=int(exceeded.sum()),
        mean_excess=excess.mean(), total_excess=excess.sum(),
        mean_excess_when_exceeded=excess[exceeded].mean() if exceeded.any() else np.nan,
        avg_target=y.mean(), avg_pred=pred.mean(),
        MAE=mean_absolute_error(y, pred), RMSE=mean_squared_error(y, pred)**0.5)

parts = []
for name, frame in [('train', train), ('test', test), ('oot', oot)]:
    part = frame[PRIMARY_KEY + [TARGET]].copy()
    part['segment'] = name
    if 'current_limit' in frame:
        part['current_limit'] = frame['current_limit']
    for label, item in results.items():
        part[f'pred_{label}'] = item['model'].predict(frame[item['selected']])
    part['gap_p99_p95'] = part['pred_p99'] - part['pred_p95']
    part['quantile_crossing'] = part['gap_p99_p95'] < 0
    parts.append(part)
scored_df = pd.concat(parts, ignore_index=True)
scored_df['month'] = pd.to_datetime(scored_df[TIME_COL]).dt.to_period('M').dt.to_timestamp()

summary_rows, month_rows, bin_rows = [], [], []
for label, item in results.items():
    for segment, group in scored_df.groupby('segment', sort=False):
        summary_rows.append(dict(model=label, segment=segment, N_features=item['best_n'],
            **quantile_metrics(group[TARGET], group[f'pred_{label}'], item['alpha'], item['baseline'])))
        for month, g in group.groupby('month'):
            month_rows.append(dict(model=label, segment=segment, month=month,
                **quantile_metrics(g[TARGET], g[f'pred_{label}'], item['alpha'], item['baseline'])))
        group = group.copy()
        # Равные прогнозы остаются в одном бине; для константного прогноза один бин.
        group['bin'] = pd.qcut(group[f'pred_{label}'], 10, labels=False, duplicates='drop')
        group['bin'] = group['bin'].fillna(0).astype(int) + 1
        for bin_id, g in group.groupby('bin'):
            bin_rows.append(dict(model=label, segment=segment, bin=bin_id,
                **quantile_metrics(g[TARGET], g[f'pred_{label}'], item['alpha'], item['baseline'])))
metrics_summary = pd.DataFrame(summary_rows)
metrics_month = pd.DataFrame(month_rows)
metrics_bins = pd.DataFrame(bin_rows)
crossing_summary = scored_df.groupby('segment').agg(
    N=('quantile_crossing', 'size'), crossing_count=('quantile_crossing', 'sum'),
    crossing_rate=('quantile_crossing', 'mean'), mean_gap=('gap_p99_p95', 'mean'),
    min_gap=('gap_p99_p95', 'min')).reset_index()
display(metrics_summary)
display(crossing_summary)

Покрытие и доля превышений указаны в долях, отклонение покрытия — в процентных пунктах. При совпадениях таргета с прогнозом покрытие может превышать номинальное. Pinball skill сравнивает модель с постоянным квантилем, рассчитанным только по train; больше — лучше. MAE/RMSE оставлены справочно. Сумма превышений — диагностический показатель, не оценка потери портфеля от снижения лимита. Для P99 особенно важно смотреть число наблюдений и превышений в каждом срезе.

In [ ]:
figures = {}
for label, item in results.items():
    monthly = metrics_month.loc[metrics_month.model == label]
    fig = px.line(monthly, x='month', y='coverage', color='segment', markers=True,
                  title=f'{label.upper()}: покрытие по месяцам')
    fig.add_hline(y=item['alpha'], line_dash='dash', annotation_text=f"Цель {item['alpha']:.0%}")
    fig.update_yaxes(tickformat='.1%')
    figures[f'{label}_coverage'] = fig
    figures[f'{label}_pinball'] = px.line(monthly, x='month', y='pinball_loss', color='segment',
        markers=True, title=f'{label.upper()}: pinball loss по месяцам')
    binned = metrics_bins.loc[metrics_bins.model == label]
    fig = px.line(binned, x='bin', y='coverage', color='segment', markers=True,
                  title=f'{label.upper()}: покрытие по бинам прогноза')
    fig.add_hline(y=item['alpha'], line_dash='dash')
    fig.update_yaxes(tickformat='.1%')
    figures[f'{label}_calibration'] = fig
    figures[f'{label}_feature_selection'] = px.line(item['metric_vs_n'], x='n_features',
        y='pinball_loss', markers=True, title=f'{label.upper()}: выбор числа признаков')
for name, fig in figures.items():
    fig.show()
    Logger.current_logger().report_plotly(title=name, series='', figure=fig)
for name, table in [('quantile_metrics', metrics_summary), ('metrics_month', metrics_month),
                    ('metrics_bins', metrics_bins), ('crossings', crossing_summary)]:
    Logger.current_logger().report_table(title=name, series='', table_plot=table)

## 9. Важности, PSI и HTML-отчёт

In [ ]:
importance_tables = {}
description_tables = {}
psi_score_tables = {}
psi_figures = {}
psi_score_figures = {}
from toolbox.modeling_utils import report

for label, item in results.items():
    model = item['model']
    feat = item['selected']
    best_features = feat
    features_sql = ','.join(f"'{f}'" for f in best_features)
    engine_sb_dirkaim = oracle.create_engine_cdw(USERNAME_SB_DIRKAIM, PASSWORD_SB_DIRKAIM)
    with engine_sb_dirkaim.connect() as conn:
        description_df = pd.read_sql(f"""select * from SB_DIRKAIM.FEATURE_REGISTRY where f_name in ({features_sql})""", conn)
    description_df = description_df[['f_name', 'f_desc', 'proc']]
    description_tables[label] = description_df
    imp = pd.Series(model.feature_importances_, index=feat).sort_values(ascending=False)
    imp_df = imp.reset_index()
    imp_df.columns = ['feature', 'importance']
    importance_tables[label] = imp_df
    display(imp_df.merge(description_df, left_on='feature', right_on='f_name', how='left'))
    pred_col = f'pred_{label}'
    psi_input = scored_df[[pred_col, 'month']].rename(columns={'month': 'year_month'})
    psi_scores = preprocessing.calculate_feat_psi(psi_input, [pred_col], date_col='year_month')
    psi_score_tables[label] = psi_scores
    _, psi_score_figures[label] = report.psi_plotly(psi_scores.T, 'PSI score')
    _, psi_figures[label] = report.psi_plotly(psi_result[feat].T)

## HTML отчет

In [ ]:
# Поля, заполняемые вручную:
TITLE = "Модель прогноза средней ежедневной задолженности за 6‑й месяц" # Название модели
DESCRIPTION = "Предсказание средней ежедневной задолженности за 6‑й месяц: дни 151–180 после даты скоринга." # Формализованное описание таргета 
TRAIN_TEST_PERIOD = "Сентябрь 2025 - январь 2026"
OOT_PERIOD = "Февраль 2026"
MODE = "offline model" # Режим работы (online/offline)
MODEL_TYPE = "LighGBM"
POPULATION_TEXT = "Клиенты имеющие КК" # Описание выборки (напр. Оформили КК или получили СМС с предложением)

In [ ]:
# Техническое имя модели
product = 'ptb_declip' # +- свободное наименование, главное отразить бизнес-процесс или бизнес-задачу, которую решает модель
monthYear = datetime.datetime.now().strftime('%m%y')

MODEL_NAME = f"{ML_TYPE}_{product}_{monthYear}_{SUBTASK}"

print(MODEL_NAME)

In [ ]:
from toolbox.modeling_utils import report

In [ ]:
model_names = {}
for label, item in results.items():
    best_model = item['model']
    imp_df = importance_tables[label]
    description_df = description_tables[label]
    psi_fig = psi_figures[label]
    psi_score_fig = psi_score_figures[label]
    MODEL_NAME = f"{ML_TYPE}_{product}_{monthYear}_{SUBTASK}_{label}"
    model_names[label] = MODEL_NAME
    # Создаём отчёт 
    report_html = report_generator.Report_HTML(
        title=f"{TITLE} — {label.upper()}",
        description=DESCRIPTION,
        footer=f"Дата создания отчёта: {datetime.date.today().strftime('%d-%m-%Y')}" 
    )

    # === ВКЛАДКА 1: СПЕЦИФИКАЦИЯ МОДЕЛИ ===
    spec_tab = report_html.add_tab("Спецификация модели")

    # Строка 1x2: Основная информация
    row1 = spec_tab.add_row("1x2")

    # Основные характеристики
    model_specs = {
        "Название": TITLE,
        "Тип задачи": ML_TYPE,
        "Квантиль": item["alpha"],
        "Режим работы": MODE,
        "Регрессия": MODEL_TYPE,
        "Версия модели": SUBTASK,
        "Разработчик модели": AUTHOR
    }
    row1.cells[0].add_kv(
        model_specs,
        title="Основные характеристики",
        caption="Ключевая информация о модели"
    )

    # Данные и выборки
    data_specs = {
        "Источник данных": OUT_TABLE,
        "Популяция": POPULATION_TEXT,
        "Период обучения": TRAIN_TEST_PERIOD,
        "Период OOT": OOT_PERIOD,
    }
    row1.cells[1].add_kv(
        data_specs,
        title="Данные и выборки", 
        caption="Информация об источниках данных и временных периодах"
    )

    # Строка 1x2: Основная метрики
    row2 = spec_tab.add_row(['1fr', '2fr'])
    model_specs = best_model.get_params()

    row2.cells[0].add_kv(
        model_specs,
        title="Параметры модели", 
        caption="Подобранные гиперпараметры"
    )

    rows = []
    for name, df in [('Train', train), ('Test', test), ('OOT', oot)]:
        y = pd.to_numeric(df[TARGET], errors='coerce')
        total = len(df)
        above_zero = (y > 0).sum()
        mean_target = y.mean()
        rows.append([
            name,
            total,
            int(above_zero),
            f'{mean_target:.4f}',  # или :.2f — как удобнее
        ])

    cnt_df = pd.DataFrame(
        rows,
        columns=['Выборка','Количество контрактов, шт.','Таргет > 0, шт.','Среднее по таргету',],
    )

    row2.cells[1].add_dataframe(
        cnt_df,
        title="Кол-во контрактов и уровень таргета", 
        caption="Информация о выборке"
    )

    # === ВКЛАДКА 2: ВЫБОРКИ ===
    sample_dfs_tab = report_html.add_tab("Данные для построения")

    # Строка 1x2: График sample
    row1 = sample_dfs_tab.add_row(['2fr', '1fr'])
    row1.cells[0].add_plotly_figure(
        fig_sample_try,
        title="Динамика таргета во времени", 
        caption="Информация о выборке"
    )
    row1.cells[1].add_dataframe(
        info_sample_try[['month', 'count', 'mean']],
        title="Динамика таргета во времени", 
        caption="Информация о выборке"
    )

    row2 = sample_dfs_tab.add_row("1x1")
    row2.cells[0].add_plotly_figure(
        fig_df ,
        title="Динамика таргета во времени", 
        # caption="Информация о Train/Test/OOT"
    )
    row3 = sample_dfs_tab.add_row("1x1")
    row3.cells[0].add_dataframe(
        info_df,
        title="Динамика таргета во времени", 
        caption="Информация о Train/Test/OOT"
    )

    # === ВКЛАДКА 3: МЕТРИКИ МОДЕЛИ ===
    model_metrics_tab = report_html.add_tab("Метрики модели")
    row1 = model_metrics_tab.add_row("1x1")
    row1.cells[0].add_dataframe(
        metrics_summary[metrics_summary['model'] == label],
        title="Квантильные метрики качества",
        caption="Pinball loss, покрытие и превышения на Train/Test/OOT. MAE/RMSE — справочно.")
    row2 = model_metrics_tab.add_row(['1fr', '2fr'])
    row2.cells[0].add_plotly_figure(figures[f'{label}_pinball'], title="Динамика Pinball loss во времени")
    row2.cells[1].add_dataframe(metrics_month[metrics_month['model'] == label], title="Метрики по месяцам")
    row3 = model_metrics_tab.add_row("1x2")
    row3.cells[0].add_plotly_figure(figures[f'{label}_coverage'], title="Покрытие по месяцам")
    row3.cells[1].add_plotly_figure(figures[f'{label}_calibration'], title="Калибровка по бинам прогноза")
    row4 = model_metrics_tab.add_row("1x3")
    for i, segment in enumerate(['train', 'test', 'oot']):
        row4.cells[i].add_dataframe(
            metrics_bins[(metrics_bins['model'] == label) & (metrics_bins['segment'] == segment)],
            title=f"Метрики по бинам {segment}")
    row5 = model_metrics_tab.add_row("1x2")
    row5.cells[0].add_plotly_figure(figures[f'{label}_feature_selection'], title="Выбор числа признаков")
    row5.cells[1].add_dataframe(crossing_summary, title="Пересечения P95/P99",
        caption="Сырые прогнозы без исправления пересечений. Gap — ширина верхнего хвоста.")

    # === ВКЛАДКА 4: ФИЧИ И PSI ===
    feat_tab = report_html.add_tab("Признаки и PSI")

    row1 = feat_tab.add_row("1x1")
    row1.cells[0].add_plotly_figure(
            psi_fig,
            title="PSI всех признаков модели по месяцам",)
    row2 = feat_tab.add_row("1x1")
    row2.cells[0].add_plotly_figure(
            psi_score_fig,
            title="PSI скоров модели по месяцам",)
    row3 = feat_tab.add_row("1x1")
    row3.cells[0].add_dataframe(
        pd.merge(imp_df, description_df, how='left', left_on='feature', right_on='f_name')[['importance', 
                                                                                          'feature', 
                                                                                          'f_desc']],
        title="FeatureImp и расшифровка признаков", 
        caption="Источник описания - SB_DIRKAIM.FEATURE_REGISTRY"
    )

    report_html.save(f'{REPORTS_FOLDER}/{MODEL_NAME}_report.html')

In [ ]:
N_PER_SEGMENT = 30
SEED = 42
demo_parts = []
for seg in ['train', 'test', 'oot']:
    part = scored_df.loc[scored_df['segment'] == seg]
    n = min(N_PER_SEGMENT, len(part))
    if n > 0:
        demo_parts.append(part.sample(n=n, random_state=SEED))
scored_df_demo = pd.concat(demo_parts, ignore_index=True)
cols_demo = ['segment'] + PRIMARY_KEY + [TARGET, 'pred_p95', 'pred_p99', 'gap_p99_p95']
scored_df_demo_view = scored_df_demo[cols_demo].round(4)
display(scored_df_demo_view)
scored_df_demo_view.to_excel('scored_df_demo_sample.xlsx', index=False)

## 10. Сохранение артефактов

In [ ]:
s3_secret = (await Secret.load("s3-retail")).get()

In [ ]:
ACCESS_KEY = s3_secret["access_key"]
SECRET_KEY = s3_secret["secret_key"]

In [ ]:
from minio import Minio

client = Minio(
    endpoint="10.167.8.142:9000",
    access_key=ACCESS_KEY,
    secret_key=SECRET_KEY,
    secure=False
)    

In [ ]:
def create_df(df, target, pred_col, y_pred, segment_name):
    result_df = df[PRIMARY_KEY+[target]].copy()
    result_df[pred_col] = y_pred
    result_df['segment'] = segment_name
    return result_df

all_psi = []
description_dfs = []
for label, item in results.items():
    target = TARGET
    model_folder = f"{PATH_MODELS_DATA}/{target}/{label}"
    os.makedirs(model_folder, exist_ok=True)
    best_features = item['selected']
    pred_col = f'{target}_pred_{label}'
    train_df = create_df(train, target, pred_col, item['model'].predict(train[best_features]), 'train')
    test_df = create_df(test, target, pred_col, item['model'].predict(test[best_features]), 'test')
    oot_df = create_df(oot, target, pred_col, item['model'].predict(oot[best_features]), 'oot')
    scored_model_df = pd.concat([train_df, test_df, oot_df], ignore_index=True)
    train[PRIMARY_KEY + best_features + [target]].to_parquet(f'{model_folder}/sample_train.parquet')
    test[PRIMARY_KEY + best_features + [target]].to_parquet(f'{model_folder}/sample_test.parquet')
    oot[PRIMARY_KEY + best_features + [target]].to_parquet(f'{model_folder}/sample_oot.parquet')
    scored_model_df.to_parquet(f'{model_folder}/scored_df.parquet')
    psi_scores = preprocessing.calculate_feat_psi(scored_model_df[[pred_col, 'report_date']], [pred_col], date_col='report_date')
    psi_scores = psi_scores.rename(columns={pred_col: 'PSI'})
    psi_scores['target'] = target
    psi_scores['quantile'] = label
    all_psi.append(psi_scores)
    psi_result[best_features].T.to_excel(f'{REPORTS_FOLDER}/psi_top20_{target}_{label}.xlsx')
    description_df = description_tables[label].copy()
    description_df['target'] = target
    description_df['quantile'] = label
    description_dfs.append(description_df)
    for segment in ['train', 'test', 'oot']:
        client.fput_object('retail',
            f'models/eservices-{TASK_NUM}/{SUBTASK}/{target}/{label}/{segment}.paruqet',
            f'{model_folder}/sample_{segment}.parquet')
    client.fput_object('retail',
        f'models/eservices-{TASK_NUM}/{SUBTASK}/{target}/{label}/scored_df.paruqet',
        f'{model_folder}/scored_df.parquet')

# Общие результаты двух квантильных моделей.
scored_df.to_parquet(f'{PATH_MODELS_DATA}/scored_quantiles.parquet')
with pd.ExcelWriter(f'{REPORTS_FOLDER}/quantile_metrics.xlsx') as writer:
    for sheet, table in [('summary', metrics_summary), ('monthly', metrics_month),
                         ('calibration', metrics_bins), ('crossings', crossing_summary)]:
        table.to_excel(writer, sheet_name=sheet, index=False)
client.fput_object('retail', f'models/eservices-{TASK_NUM}/{SUBTASK}/scored_quantiles.parquet',
                   f'{PATH_MODELS_DATA}/scored_quantiles.parquet')

In [ ]:
psi_pred = pd.concat(all_psi, ignore_index=True)
descriptions = pd.concat(description_dfs, ignore_index=True)

## 12. Публикация модели в Model_Registry

In [ ]:
# Креды для подключения S3
s3_kwargs = (await Secret.load("s3-retail")).get()
# Пароль для работы с SB_DIRKAIM (!)
# cdw_pass = (await Secret.load("pass-sb-dirkaim")).get() 


from model_repository import ModelRepository
from model_repository.dao import OracleDAO
from model_repository.storage import MinioClient

engine = oracle.create_engine_cdw(
    USERNAME_SB_DIRKAIM, PASSWORD_SB_DIRKAIM
)
dao = OracleDAO(engine)
storage = MinioClient(**s3_kwargs)

# Инициализируем класс ModelRepository
repo = ModelRepository(storage=storage, dao=dao)

In [ ]:
from model_repository import RegisterRequest

registered_models = {}
for label, item in results.items():
    MODEL_NAME = model_names[label]
    best_model = item["model"]
    req = RegisterRequest(
        model_name=MODEL_NAME,
        model_type="lightgbm",  # "catboost" | "lightgbm" | "sklearn" | "other"
        artifact_format="pickle",
        uri=f"retail/models/eservices-{TASK_NUM}/{SUBTASK}/{MODEL_NAME}.pkl",
        created_by=AUTHOR,
        description=f"{label.upper()}: квантиль ОД по контракту на 6 месяц", # Описание целевой переменной
        scoring_flg=0,  # Флаг что модель на активном скоринге
        monitoring_flg=0,  # Флаг что модель на мониторинге
        is_online=0,  # Флаг онлайн модели
        model_obj=best_model,  # авто-сериализуем, авто-извлечём важности
    )
    meta = repo.register(req)
    registered_models[label] = meta
    print(meta)
    assert repo.exists(MODEL_NAME)